In [ ]:
%reset -f
from main import cost_func, aero_cost
import pygad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# Starting variables
wingspan = 1.9422 # 1 m
mid_chord = 0.10577 # 15 cm
tip_chord = 0.07833 # 15 cm
w_flange = 0.005565
t_flange = 0.0067967
t_web = 0.0074199
t_skin_root = 0.000938 # 1 mm
t_skin_mid = 0.00031859
t_skin_tip = 0.00148899
mid_twist = -0.901387 # 0 deg
tip_twist = -0.690297 # 0 deg
skin_index = 1
spar_index = 1

# # Hard-code these values into cost equation
# spar_volume = 0.303*0.15*0.05*wingspan
# S = wing_area(wingspan, mid_chord, tip_chord)
# skin_volume = 2*S*t_skin
# V_maxR, Preq, _, _ = aero(wingspan, mid_chord, tip_chord, root_twist, mid_twist, tip_twist, spar_mat, skin_mat, spar_volume, skin_volume, 0, 0)
# desired_range = range(V_maxR, Preq)
# desired_cost = price(spar_mat, skin_mat, spar_volume, skin_volume)



In [ ]:
init_best = [wingspan, mid_chord, tip_chord, w_flange, t_flange, t_web, t_skin_root, t_skin_mid, t_skin_tip, mid_twist, tip_twist, skin_index, spar_index]

In [ ]:
def fitness_func(ga_instance, solution, solution_idx):
    cost, range_est, mass_est, tip_def = cost_func(solution[0], solution[1], solution[2], solution[3], solution[4], solution[5], solution[6], solution[7], solution[8], solution[9], solution[10], solution[11], solution[12])
    fitness = -cost

    # if not hasattr(ga_instance, "last_outputs"):
    #     ga_instance.last_outputs = []

    # ga_instance.last_outputs.append((fitness, range_est, mass_est, tip_def))

    return fitness


In [ ]:
# population_history = []

def on_gen(ga_instance):

    # solution, fitness, idx = ga_instance.best_solution()
    # population_history.append(ga_instance.population.copy())
    # fig, ax1 = plt.subplots(1,1,figsize=(10,5))

    if ga_instance.generations_completed % 1 == 0:
        solution, fitness, _ = ga_instance.best_solution()

        cost_best, range_best, mass_best, deflect_tip = cost_func(*solution)

        print(f"Gen {ga_instance.generations_completed} | Best Fitness = {fitness}")
        print(f"Range: {range_best:.2f} km, Mass: {mass_best:.2f} kg")
        print("Design Variables:")
        print(f"Wingspan: {solution[0]}")
        print(f"Mid chord: {solution[1]}")
        print(f"Tip chord: {solution[2]}")
        print(f"Flange width: {solution[3]}")
        print(f"Flange thickness: {solution[4]}")
        print(f"Web thickness: {solution[5]}")
        print(f"Skin root thickness: {solution[6]}")
        print(f"Skin mid thickness: {solution[7]}")
        print(f"Skin tip thickness: {solution[8]}")
        print(f"Mid twist: {solution[9]}")
        print(f"Tip twist: {solution[10]}")
        print(f"Skin material: {solution[11]}")
        print(f"Spar material: {solution[12]}")
        print(f"Intermediate Variables:")
        print(f"Tip Deflection: {deflect_tip} deg")

        # ax1.scatter(ga_instance.generations_completed, fitness)


In [ ]:
gene_space = [
    {'low':0.3, 'high':10}, # wingspan
    {'low':0.1, 'high':0.15}, # mid chord
    {'low':0.05, 'high':0.1}, # tip chord
    {'low':1e-5, 'high':7.575e-3}, # flange_width
    {'low':1e-5, 'high':7.575e-3}, # flange_thickness
    {'low':1e-5, 'high':7.575e-3}, # web_thickness
    {'low':1e-5, 'high':7.575e-3}, # skin thickness
    {'low':1e-5, 'high':5.05e-3}, # skin thickness
    {'low':1e-5, 'high':2.525e-3}, # skin thickness
    {'low':-5, 'high':0}, # mid twist
    {'low':-5, 'high':0}, # tip twist
    [0,1,2,3], # skin material
    [0,1,2,3] # spar material
]

gene_type = [
    float, # wingspan
    float, # mid chord
    float, # tip chord
    float, # flange width
    float, # flange thickness
    float, # flange thickness
    float, # flange thickness
    float, # web thickness
    float, # skin thickness
    float, # mid twist
    float, # tip twist
    int, # skin material
    int # spar material
]

sol_per_pop = 30
num_genes = len(init_best)

init_pop = []

for _ in range(sol_per_pop):
    individual = []

    for i, val in enumerate(init_best):
        space = gene_space[i]

        # Continuous variables
        if isinstance(space, dict):
            low, high = space['low'], space['high']

            # Add small perturbation (5% of range)
            perturb = 0.05 * (high - low)
            new_val = val + np.random.uniform(-perturb, perturb)

            # Clip to bounds
            new_val = np.clip(new_val, low, high)

        # Discrete variables
        else:
            # Keep same material most of the time, sometimes mutate
            if np.random.rand() < 0.8:
                new_val = val
            else:
                new_val = np.random.choice(space)

        individual.append(new_val)

    init_pop.append(individual)

init_pop = np.array(init_pop)

# Force exact base design into population
init_pop[0] = init_best

# init_pop[0] = init_best

ga_inst = pygad.GA(num_generations=100, # number of generations run
                num_parents_mating=8, # how many of best solutions are selected to be parents in the next, roughly half of or less than half of sol_per_pop
                sol_per_pop=30, # each population has this many solutions, only meaningful if initial_population is not used, small value might prevent finding high-quality solutions, large value might lead to unnecessary computational expense
                keep_parents=1, # contols elitism, how many parents from current generation are carried to the next, typically 1 or 2
                num_genes=len(init_best), # each solution has this many genes with constraints set by gene_space
                fitness_func=fitness_func,
                initial_population=init_pop,
                parent_selection_type="tournament",
                K_tournament=3,
                crossover_type="uniform",
                mutation_type="random",
                mutation_percent_genes=30, # increased from 25
                # mutation_type="adaptive",
                # mutation_num_genes=[5,1],
                gene_space=gene_space, # use instead of init_range_low/high, constraints to genes
                gene_type=gene_type, # value type for genes
                on_generation=on_gen,
                parallel_processing=7)

ga_inst.run()

# try with existing or known result

In [ ]:
plt.plot(ga_inst.best_solutions_fitness)
plt.xlabel("Generation")
plt.ylabel("Best Fitness")
plt.title("GA Convergence")
plt.show()

In [ ]:
# solution, fitness, _ = ga_inst.best_solution()

In [ ]:
ga_inst.save(filename="run_9")

In [ ]:
# population_history = np.array(population_history)